# P2 · Fleet Maintenance Planning

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
INSPECTION, SHORTAGE, SURPLUS = 10.0, 0.20, 0.02
SENSORS = [f"s{i:03d}" for i in range(170)]
SCHEMA = ["id", "flag", "budget"]

def find_data():
    required = {"train.csv", "test.csv", "sample_submission.csv"}
    kaggle = Path("/kaggle/input")
    candidates = [Path.cwd(), *(p.parent for p in sorted(kaggle.rglob("train.csv")))]
    matches = []
    for folder in dict.fromkeys(candidates):
        if all((folder / name).is_file() for name in required):
            test_cols = set(pd.read_csv(folder / "test.csv", nrows=0).columns)
            train_cols = set(pd.read_csv(folder / "train.csv", nrows=0).columns)
            if {"id", *SENSORS} <= test_cols and {"failure", "repair_cost", *SENSORS} <= train_cols:
                matches.append(folder)
    if len(matches) != 1:
        raise ValueError(f"Expected one P2 dataset. Matching directories: {matches}")
    return matches[0]

data = find_data()
train = pd.read_csv(data / "train.csv", dtype={"id": str})
test = pd.read_csv(data / "test.csv", dtype={"id": str})
sample = pd.read_csv(data / "sample_submission.csv", dtype={"id": str})
assert len(train) == 56_000 and len(test) == len(sample) == 20_000
assert list(sample) == SCHEMA
assert test.id.is_unique and sample.id.is_unique and set(test.id) == set(sample.id)
assert train.failure.isin([0, 1]).all()
failure = train.failure.to_numpy(dtype=int)
repair = train.repair_cost.fillna(0).to_numpy()
assert np.isfinite(repair).all() and np.array_equal(repair > 0, failure == 1)
print(f"{len(train):,} training vehicles, {failure.sum():,} recorded failures")

## Sensor readings

The test channels `s055`–`s063` match training channels `s054`–`s062` after a one-column shift. Scale them to a common resolution. Use stable counters to measure each reading relative to the vehicle’s overall sensor level.

In [ ]:
shift = {}
for name in SENSORS:
    old, new = train[name].dropna(), test[name].dropna()
    if min(old.nunique(), new.nunique()) < 2:
        shift[name] = 1.0
    else:
        distribution_shift = ks_2samp(old, new, method="asymp").statistic
        missingness_shift = abs(len(old) / len(train) - len(new) / len(test))
        shift[name] = max(distribution_shift, missingness_shift)

stable = [name for name in SENSORS if shift[name] < 0.10]

frequently_missing = train[stable].isna().mean().to_numpy() >= 0.20

def features(frame):
    readings = frame[stable].to_numpy(dtype="float32")
    missing = np.isnan(readings)
    counts = missing.sum(axis=1, dtype="float32")
    group_counts = missing[:, frequently_missing].sum(axis=1, dtype="float32")
    return np.column_stack((readings, counts, group_counts))

X, X_test = features(train), features(test)
print(f"Stable raw sensors: {len(stable)}")

# Normalize each reading by its training-depot scale before computing ratios.
scale = train[stable].quantile(0.90).to_numpy()
scale = np.where(scale > 0, scale, 1.0)
anchor_names = ["s071", "s123", "s149", "s053", "s012"]
anchor_cols = [stable.index(c) for c in anchor_names if c in stable]

# Adjacent columns match across depots after a one-column offset.
source_channels = [f"s{i:03d}" for i in range(54, 63)]
target_channels = [f"s{i:03d}" for i in range(55, 64)]
resolution = np.array([3, 10, 10, 100, 100, 100, 10, 10, 10], dtype="float32")
channel_scale = np.maximum(train[source_channels].quantile(0.9).to_numpy(dtype="float32"), 1)

def relative_features(frame, channels):
    raw = frame[stable].to_numpy(dtype="float32")
    scaled = raw / scale.astype("float32")
    exposure = pd.DataFrame(scaled[:, anchor_cols]).median(axis=1).to_numpy()
    log_readings = np.log1p(scaled)
    ratios = np.log1p(scaled / (0.01 + exposure[:, None]))
    parts = [log_readings, ratios, features(frame)[:, -2:]]
    # These blocks have the same missingness pattern and adjacent channels.
    for start, end in [(23, 33), (37, 47), (80, 90), (109, 119), (129, 139), (154, 164)]:
        cols = [stable.index(f"s{i:03d}") for i in range(start, end) if f"s{i:03d}" in stable]
        if len(cols) < 2:
            continue
        block = scaled[:, cols]
        total = np.sum(block, axis=1)
        parts.append(block / (0.01 + total[:, None]))
    block = frame[channels].to_numpy(dtype="float32")
    block = np.rint(block / resolution) * resolution / channel_scale
    parts.extend([np.log1p(block), np.log1p(block / (0.01 + exposure[:, None]))])
    return np.column_stack(parts).astype("float32")

Z, Z_test = relative_features(train, source_channels), relative_features(test, target_channels)

def active_channel_ratio(frame, name):
    active = frame[name].gt(100) & frame["s149"].gt(0)
    return frame.loc[active, name] / (1.0 + frame.loc[active, "s149"])

aligned_shift = [
    ks_2samp(active_channel_ratio(train, old), active_channel_ratio(test, new), method="asymp").statistic
    for old, new in zip(source_channels, target_channels)
]
print(f"Restored {len(aligned_shift)} channels. Largest active-ratio KS distance: {max(aligned_shift):.3f}")

## Failure risk and repair cost

Fit boosted trees and extra trees for failure risk. Fit boosted trees and sparse regression on failed vehicles to estimate log repair cost.

In [ ]:
def fit_predict(fit_idx, query, seed):
    x, y, c = X[fit_idx], failure[fit_idx], repair[fit_idx]
    positive = y == 1
    cls_options = dict(learning_rate=0.075, max_iter=210, max_bins=127,
                       early_stopping=False, random_state=seed)
    probabilities = []
    for leaves, min_leaf in ((7, 25), (15, 40)):
        clf = HistGradientBoostingClassifier(
            **cls_options, max_leaf_nodes=leaves, min_samples_leaf=min_leaf,
            l2_regularization=3.0, max_features=0.85)
        clf.fit(x, y)
        probabilities.append(clf.predict_proba(query)[:, 1])
    p = np.mean(probabilities, axis=0)

    mean_cost = c[positive].mean()
    weights = np.where(positive, c / mean_cost, 1.0)
    weighted = HistGradientBoostingClassifier(
        **cls_options, max_leaf_nodes=7, min_samples_leaf=35,
        l2_regularization=5.0, max_features=0.85)
    weighted.fit(x, y, sample_weight=weights)
    p_weighted = np.clip(weighted.predict_proba(query)[:, 1], 0, 0.985)
    expected_cost = mean_cost * p_weighted / (1 - p_weighted) * (1 - p)

    severity = HistGradientBoostingRegressor(
        max_leaf_nodes=5, min_samples_leaf=20, max_iter=65,
        learning_rate=0.06, l2_regularization=4.0, early_stopping=False,
        random_state=seed)
    severity.fit(x[positive], np.log(c[positive]))
    return np.column_stack((p, expected_cost, severity.predict(query)))


def cross_fit(indices, query, seed, n_splits):
    oof = np.empty((len(indices), 3))
    predictions = []
    split_y = failure[indices]
    splitter = StratifiedKFold(n_splits, shuffle=True, random_state=seed)
    for fold, (fit, valid) in enumerate(splitter.split(indices, split_y), start=1):
        combined = np.vstack((X[indices[valid]], query))
        output = fit_predict(indices[fit], combined, seed + fold)
        oof[valid] = output[:len(valid)]
        predictions.append(output[len(valid):])
        print(f"Cross-fit fold {fold}/{n_splits}", flush=True)
    return oof, np.mean(predictions, axis=0)

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler

def extra_predictions(indices, query, seed):
    x, y, c = Z[indices], failure[indices], repair[indices]
    positive = y == 1
    # Sample healthy vehicles for the forest, then restore the original class odds.
    rng = np.random.default_rng(seed)
    healthy = np.flatnonzero(~positive)
    selected = rng.choice(healthy, size=min(11000, len(healthy)), replace=False)
    rows = np.r_[np.flatnonzero(positive), selected]
    forest = ExtraTreesClassifier(n_estimators=200, min_samples_leaf=2, max_features=0.22,
                                  n_jobs=4, random_state=seed)
    forest.fit(np.nan_to_num(x[rows], nan=-1), y[rows])
    score = forest.predict_proba(np.nan_to_num(query, nan=-1))[:, 1]
    keep = len(selected) / len(healthy)
    p_forest = keep * score / (1 - score + keep * score)

    boost = HistGradientBoostingClassifier(max_leaf_nodes=12, min_samples_leaf=30, max_iter=175,
                                           max_bins=127, learning_rate=0.065, max_features=0.7,
                                           l2_regularization=6, early_stopping=False, random_state=seed)
    boost.fit(x, y, sample_weight=np.where(positive, 6.0, 1.0))
    score = boost.predict_proba(query)[:, 1]
    p_boost = score / (6.0 - 5.0 * score)

    # Sparse log-cost regression on the failed vehicles.
    def severity_features(rows):
        return np.column_stack((np.nan_to_num(rows, nan=0), np.isnan(rows), rows > 0))
    scaler = StandardScaler()
    design = scaler.fit_transform(severity_features(x[positive]))
    model = ElasticNet(alpha=0.09, l1_ratio=0.55, max_iter=4000)
    model.fit(design, np.log(c[positive]))
    log_cost = model.predict(scaler.transform(severity_features(query)))
    log_cost = np.clip(log_cost, np.log(c[positive].min()), np.log(c.max()))
    return np.column_stack((p_forest, p_boost, log_cost))


def extra_cross_fit(indices, query, seed, n_splits):
    oof = np.empty((len(indices), 3))
    predictions = []
    splitter = StratifiedKFold(n_splits, shuffle=True, random_state=seed)
    for fold, (fit, valid) in enumerate(splitter.split(indices, failure[indices]), start=1):
        output = extra_predictions(indices[fit], np.vstack((Z[indices[valid]], query)), seed + fold)
        oof[valid] = output[:len(valid)]
        predictions.append(output[len(valid):])
        print(f"Relative-sensor models, fold {fold}/{n_splits}", flush=True)
    return oof, np.mean(predictions, axis=0)

## Inspection and parts budget

Calibrate failure probabilities and repair costs using out-of-fold predictions. Inspect when expected savings cover the inspection and parts fees. The optimal budget is the 10/11 quantile of estimated parts demand.

In [ ]:
def repair_distribution(prediction, reference, reference_cost, conditional):
    positive = reference_cost > 0
    observed = reference_cost[positive]
    if conditional:
        residuals = np.exp(np.log(observed) - reference[positive, 2])
        scale = np.exp(prediction[:, 2])
    else:
        residuals = observed
        scale = np.ones(len(prediction))
    values = np.sort(residuals)
    cumulative = np.r_[0.0, np.cumsum(values)]
    return scale, values, cumulative


def decisions(prediction, distribution, probability_scale, direct_share):
    scale, values, cumulative = distribution
    conditional_mean = scale * values.mean()
    direct_risk = prediction[:, 1] / conditional_mean
    source_p = np.clip(
        (1 - direct_share) * prediction[:, 0] + direct_share * direct_risk, 0.0, 1.0
    )
    p = probability_scale * source_p / (1 + (probability_scale - 1) * source_p)
    fraction = 1 - SURPLUS / ((SHORTAGE + SURPLUS) * np.maximum(p, 1e-12))
    index = np.clip(np.ceil(fraction * len(values)).astype(int) - 1, 0, len(values) - 1)
    budget = np.where(fraction > 0, np.rint(scale * values[index]), 0.0)
    below = np.searchsorted(values, budget / scale, side="right")
    surplus = np.maximum(0.0, (budget * below - scale * cumulative[below]) / len(values))
    shortage = np.maximum(0.0, conditional_mean - budget + surplus)
    inspect_cost = INSPECTION + SHORTAGE * p * shortage + SURPLUS * ((1 - p) * budget + p * surplus)
    flag = p * conditional_mean > inspect_cost
    return flag.astype(int), np.where(flag, budget, 0).astype(int)


def total_cost(actual, flag, budget):
    inspected = INSPECTION + SHORTAGE * np.maximum(actual - budget, 0) + SURPLUS * np.maximum(budget - actual, 0)
    return np.where(flag == 1, inspected, actual).sum()


assert total_cost(np.array([400, 0, 1200, 0]), np.array([1, 1, 0, 0]),
                  np.array([300, 50, 0, 0])) == 1241

def estimate_prior(indices):
    """Fit a two-class mixture to high-risk tails of stable sensors."""
    old = X[indices, :len(stable)]
    new = X_test[:, :len(stable)]
    y = failure[indices].astype(bool)
    candidates = []
    for col, name in enumerate(stable):
        if shift[name] >= 0.06:
            continue
        positive, negative = old[y, col], old[~y, col]
        if np.isfinite(positive).sum() < 0.85 * y.sum():
            continue
        direction = 1 if np.nanmedian(positive) >= np.nanmedian(negative) else -1
        cutoffs = np.nanquantile(direction * negative, [0.95, 0.97, 0.98, 0.99])
        positive_tail = (direction * positive[:, None] > cutoffs).mean(axis=0)
        negative_tail = (direction * negative[:, None] > cutoffs).mean(axis=0)
        test_tail = (direction * new[:, col, None] > cutoffs).mean(axis=0)
        contrast = positive_tail - negative_tail
        if contrast[0] >= 0.55:
            estimate = np.median((test_tail - negative_tail) / np.maximum(contrast, 1e-6))
            candidates.append((contrast[0], name, estimate))
    # Use the strongest class separators and a median to limit the effect of shifted sensors.
    top = sorted(candidates, reverse=True)[:12]
    if len(top) < 3:
        return y.mean(), top
    return float(np.clip(np.median([estimate for _, _, estimate in top]), 0.002, 0.15)), top


def prior_ratio(source, target):
    return (target / (1 - target)) / (source / (1 - source))


def vehicle_weights(actual, target, source):
    return np.where(actual > 0, target / source, (1 - target) / (1 - source))


def individual_cost(actual, flag, budget):
    return np.where(flag, INSPECTION + SHORTAGE * np.maximum(actual - budget, 0)
                    + SURPLUS * np.maximum(budget - actual, 0), actual)



from scipy.special import logit
from sklearn.linear_model import LogisticRegression

# Weights are ordered as raw boosting, relative-reading boosting and extra trees.
risk_models = {
    "raw": (1.0, 0.0, 0.0),
    "boost blend": (0.45, 0.55, 0.0),
    "three-model blend": (0.35, 0.40, 0.25),
    "relative sensors": (0.0, 0.70, 0.30),
}

def combine(raw, extra, family):
    a, b, c = risk_models[family]
    p = a * raw[:, 0] + b * extra[:, 1] + c * extra[:, 0]
    return np.column_stack((p, raw[:, 1], raw[:, 2], extra[:, 2]))


def calibrated_demand(pred, reference, actual, target_prior):
    observed = actual > 0
    def log_odds(p):
        return logit(np.clip(p, 1e-7, 1 - 1e-7)).reshape(-1, 1)

    calibration = LogisticRegression(C=1)
    calibration.fit(log_odds(reference[:, 0]), observed)
    source_p = calibration.predict_proba(log_odds(pred[:, 0]))[:, 1]

    # Calibrate the spread of log-cost predictions using only held-out failures.
    y = np.log(actual[observed])
    # Learn a convex blend from held-out log costs. Shrink its weight toward equal shares.
    tree = reference[observed, 2]
    difference = reference[observed, 3] - tree
    weight = np.clip((difference @ (y - tree) + 5) / (difference @ difference + 10), 0, 1)
    x = tree + weight * difference
    severity = pred[:, 2] + weight * (pred[:, 3] - pred[:, 2])
    slope = np.clip((np.dot(x - x.mean(), y - y.mean()) + 20) /
                    (np.dot(x - x.mean(), x - x.mean()) + 20), 0.5, 1.75)
    center = np.exp(np.clip(y.mean() + slope * (severity - x.mean()), -4, 11))
    residuals = np.sort(np.exp(y - y.mean() - slope * (x - x.mean())))
    prefix = np.r_[0, np.cumsum(residuals)]
    ceiling = actual.max()
    limit = np.searchsorted(residuals, ceiling / center, side="right")
    mean_cost = (center * prefix[limit] + ceiling * (len(residuals) - limit)) / len(residuals)
    odds = prior_ratio(observed.mean(), target_prior)
    return source_p, odds, center, residuals, prefix, ceiling, mean_cost


def calibrated_policy(demand, expected_raw, multiplier):
    source_p, odds, center, residuals, prefix, ceiling, mean_cost = demand
    # The cost-weighted classifier supplies a second estimate of the skip loss.
    # Keep the budget tied to the calibrated demand distribution.
    adjusted_odds = odds * multiplier
    p = adjusted_odds * source_p / (1 + (adjusted_odds - 1) * source_p)
    rate = SURPLUS / (SHORTAGE + SURPLUS)
    q = 1 - rate / np.maximum(p, 1e-12)
    ix = np.clip(np.ceil(q * len(residuals)).astype(int) - 1, 0, len(residuals) - 1)
    budget = np.where(q > 0, np.minimum(center * residuals[ix], ceiling), 0.0)
    below = np.searchsorted(residuals, budget / center, side="right")
    positive_excess = np.maximum((budget * below - center * prefix[below]) / len(residuals), 0)
    shortage = np.maximum(mean_cost - budget + positive_excess, 0)
    inspect = INSPECTION + SHORTAGE * p * shortage + SURPLUS * ((1-p)*budget + p*positive_excess)
    direct = expected_raw * p / np.maximum(source_p, 1e-7)
    skip = 0.8 * p * mean_cost + 0.2 * direct
    flag = skip > inspect
    return flag.astype(int), np.where(flag, np.round(budget, 2), 0.0)


def choose_policy(raw, extra, actual, folds, target_prior):
    rates = np.clip(target_prior * np.array([0.65, 1.0, 1.35]), 0.001, 0.25)
    records = []
    for family in risk_models:
        pred = combine(raw, extra, family)
        fold_demand = [calibrated_demand(pred[val], pred[fit], actual[fit], target_prior)
                       for fit, val in folds]
        for multiplier in (0.8, 1.0, 1.25):
            losses = np.empty(len(actual))
            for (_, val), demand in zip(folds, fold_demand):
                flags, budgets = calibrated_policy(demand, pred[val, 1], multiplier)
                losses[val] = individual_cost(actual[val], flags, budgets)
            costs = [np.average(losses, weights=vehicle_weights(actual, rate, (actual > 0).mean()))
                     for rate in rates]
            records.append((family, multiplier, *costs))
    report = pd.DataFrame(records, columns=["family", "multiplier", "low", "central", "high"])
    costs = report[["low", "central", "high"]]
    report["objective"] = (costs / costs.min()).mean(axis=1)
    best = report.loc[report.objective.idxmin()]
    return best.family, best.multiplier, report

## Validation

Select the model blend on 42,000 vehicles. Compare it with the previous submission’s policy on the other 14,000, using the same cost formula and assumed failure rates.

In [ ]:
development, audit = train_test_split(np.arange(len(train)), test_size=0.25,
                                      stratify=failure, random_state=SEED)
dev_raw, audit_raw = cross_fit(development, X[audit], SEED, n_splits=4)
dev_extra, audit_extra = extra_cross_fit(development, Z[audit], SEED, n_splits=4)
dev_cost = repair[development]
dev_prior, _ = estimate_prior(development)
folds = list(StratifiedKFold(4, shuffle=True, random_state=SEED).split(development, failure[development]))
family, multiplier, report = choose_policy(dev_raw, dev_extra, dev_cost, folds, dev_prior)
print(report.sort_values("objective").round(3).head().to_string(index=False))

ref = combine(dev_raw, dev_extra, family)
query = combine(audit_raw, audit_extra, family)
demand = calibrated_demand(query, ref, dev_cost, dev_prior)
new_flag, new_budget = calibrated_policy(demand, query[:, 1], multiplier)
old_distribution = repair_distribution(audit_raw, dev_raw, dev_cost, True)
old_odds = 1.35 * prior_ratio(failure[development].mean(), dev_prior)
old_flag, old_budget = decisions(audit_raw, old_distribution, old_odds, 0.35)
rows = []
for rate in [failure[development].mean(), *np.array([0.65, 1, 1.35]) * dev_prior]:
    weights = vehicle_weights(repair[audit], rate, failure[audit].mean())
    old_cost = np.average(individual_cost(repair[audit], old_flag, old_budget), weights=weights)
    new_cost = np.average(individual_cost(repair[audit], new_flag, new_budget), weights=weights)
    rows.append((rate, old_cost, new_cost))
print(pd.DataFrame(rows, columns=["assumed failure rate", "previous cost/vehicle",
                                "new cost/vehicle"]).round(4).to_string(index=False))

## Submission

Refit on all training data and write `/kaggle/working/submission.csv` on Kaggle.

In [ ]:
indices = np.arange(len(train))
all_raw, test_raw = cross_fit(indices, X_test, SEED + 1, n_splits=5)
all_extra, test_extra = extra_cross_fit(indices, Z_test, SEED + 1, n_splits=5)
ref = combine(all_raw, all_extra, family)
query = combine(test_raw, test_extra, family)
target_prior, _ = estimate_prior(indices)
demand = calibrated_demand(query, ref, repair, target_prior)
flag, budget = calibrated_policy(demand, query[:, 1], multiplier)
predictions = pd.DataFrame({"id": test.id, "flag": flag, "budget": budget}).set_index("id")
submission = predictions.loc[sample.id].reset_index()

output_dir = Path("/kaggle/working")
if not output_dir.is_dir():
    output_dir = Path.cwd()
output = output_dir / "submission.csv"
submission.to_csv(output, index=False)
written = pd.read_csv(output, dtype={"id": str})
assert written.columns.tolist() == SCHEMA
assert len(written) == 20_000 and written.id.is_unique and written.id.equals(sample.id)
assert written.flag.isin([0, 1]).all()
assert np.isfinite(written.budget).all() and written.budget.ge(0).all()
assert written.loc[written.flag.eq(0), "budget"].eq(0).all()
print(f"Saved {len(written):,} rows to {output}")
print(f"Inspections: {written.flag.sum():,}. Positive budgets: {written.budget.gt(0).sum():,}.")
print(written.head().to_string(index=False))